# NB25 · Descriptors & Metaprogramming

`advanced/` · notebook 25 of 32 · 32 questions + a mini-project

This notebook opens up the machinery under every class you've written. What
really happens when you read `obj.x`? How does `@property` work? Who builds a
class when Python reaches the end of a `class` block? And how can
`isinstance` be made to answer a question *you* choose?

The answers come down to a few small protocols: attribute lookup with
`__getattr__`, `__getattribute__` and `__setattr__`, *descriptors*
(`__get__`, `__set__`, `__set_name__`), class creation with `type()`, and
metaclasses. Frameworks like Django and SQLAlchemy are built on them.

Use these tools sparingly. Most code never needs a metaclass. The real goal
is to understand what frameworks do for you, and to recognise when a simpler
tool (a `property`, a descriptor, `__init_subclass__`) does the job.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB24: context managers
- [ ] NB21: advanced OOP

## What you'll practise

- Attribute lookup order; __getattr__ vs __getattribute__; __setattr__
- Descriptors: __get__, __set__, __set_name__; data vs non-data descriptors
- How property, methods and classmethod are built on descriptors
- __init_subclass__ and __class_getitem__
- Creating classes with type(); metaclasses (__new__, __call__)
- inspect, vars(), dir(); monkey-patching; why eval and exec are dangerous
- __instancecheck__, __subclasshook__ and ABC.register: how isinstance can be customised

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB25-Q01 · L1 · __getattr__ · Write

Write a class `Settings` whose `__init__` sets `self.theme = "dark"`. Add
a `__getattr__` method so that reading any attribute that doesn't exist
gives `"unset"` instead of an error. Create one instance and print its
`theme` and its `font` on one line:

```text
dark unset
```

<details><summary><b>Hint</b></summary>

Python calls `__getattr__(self, name)` only when the normal lookup has already failed. Whatever it returns becomes the attribute's value.

</details>

<details><summary><b>Expected output</b></summary>

```text
dark unset
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Settings:
    def __init__(self):
        self.theme = "dark"

    def __getattr__(self, name):
        return "unset"


settings = Settings()
print(settings.theme, settings.font)
```

Normal lookup finds `theme` in the instance's `__dict__`, so `__getattr__` never runs for it. `font` isn't on the instance, the class or any parent, so Python calls `__getattr__(settings, "font")` as a last resort and uses whatever it returns.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Settings:
    def __init__(self):
        self.theme = "dark"

    def __getattr__(self, name):
        if name.startswith("__"):
            raise AttributeError(name)
        return "unset"


settings = Settings()
print(settings.theme, settings.font)
```

A catch-all `__getattr__` also answers questions that libraries ask quietly. `copy.deepcopy` runs `getattr(x, "__deepcopy__", None)`, gets `"unset"` back, tries to call it and crashes. Seniors refuse dunder names with an honest `AttributeError`, so those probes keep working.

</details>

In [ ]:
# your code here


### NB25-Q02 · L1 · vars() and dir() · Predict

`vars()` on an instance. Predict the output.

**What does this print?**

```python
class Sensor:
    unit = "C"

    def __init__(self, name):
        self.name = name


probe = Sensor("probe-1")
probe.reading = 21.5
print(vars(probe))
print("unit" in vars(probe), probe.unit)
```

<details><summary><b>Hint</b></summary>

`vars(obj)` returns the object's own `__dict__`. Where does `unit` live?

</details>

<details><summary><b>Expected output</b></summary>

```text
{'name': 'probe-1', 'reading': 21.5}
False C
```

</details>

<details><summary><b>Solution</b></summary>

`vars(probe)` is the instance's own `__dict__`: only the attributes stored *on that object*, in the order they were added. `name` came from `__init__` and `reading` was added later. `unit` lives on the class, so it isn't in the instance's dict, yet `probe.unit` still works because lookup falls back to the class.

</details>

<details><summary><b>Senior dev solution</b></summary>

`vars()` is the quickest way to see an object's real state while debugging. It raises `TypeError` for objects without a `__dict__`, such as ints or classes using `__slots__` (NB21).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB25-Q03 · L1 · Creating classes with type() · Write

Without a `class` statement, create a class called `Point` using `type()`.
It should have two class attributes, `x = 0` and `y = 0`. Print the
class's name and both attributes:

```text
Point 0 0
```

<details><summary><b>Hint</b></summary>

With three arguments, `type(name, bases, namespace)` builds a class: a name string, a tuple of base classes (empty here) and a dict of attributes.

</details>

<details><summary><b>Expected output</b></summary>

```text
Point 0 0
```

</details>

<details><summary><b>Solution</b></summary>

```python
Point = type("Point", (), {"x": 0, "y": 0})
print(Point.__name__, Point.x, Point.y)
```

With one argument, `type(obj)` tells you an object's class. With three, it *builds* a class. A `class` statement does exactly this behind the scenes: it runs the body to fill a namespace dict, then calls `type(name, bases, namespace)`. `type` is the class of classes.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Point:
    x = 0
    y = 0


print(Point.__name__, Point.x, Point.y)
```

When you know the class while writing the code, the `class` statement is clearer. Seniors use `type()` only when a class's shape comes from data at runtime (a database schema, a config file), as in Q31.

</details>

In [ ]:
# your code here


---

## L2 · Easy

### NB25-Q04 · L2 · Descriptors · Write

Write a descriptor class `Area` with a `__get__` method that returns the
instance's `width * height`. Use it in a class `Room` as `area = Area()`;
`Room.__init__` stores `width` and `height`.

Print the area of a 3 by 4 room, then change its `width` to `5` and print
the area again:

```text
12
20
```

<details><summary><b>Hint</b></summary>

`__get__(self, obj, objtype=None)` receives the descriptor as `self` and the instance being read from as `obj`.

</details>

<details><summary><b>Expected output</b></summary>

```text
12
20
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Area:
    def __get__(self, obj, objtype=None):
        return obj.width * obj.height


class Room:
    area = Area()

    def __init__(self, width, height):
        self.width = width
        self.height = height


room = Room(3, 4)
print(room.area)
room.width = 5
print(room.area)
```

When lookup finds `area` on the class and that object has a `__get__` method, Python calls `Area.__get__(descriptor, room, Room)` and returns the result instead of the descriptor itself. It's worked out fresh on every read, so it follows the change to `width`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Room:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height


room = Room(3, 4)
print(room.area)
room.width = 5
print(room.area)
```

`property` *is* a descriptor, ready-made (Q24 rebuilds it). For one computed value on one class, `@property` is the right tool. Write a descriptor class when the same behaviour repeats across many attributes or classes, as in Q19.

</details>

In [ ]:
# your code here


### NB25-Q05 · L2 · __set_name__ · Write

Write a class `Field` with only one method, `__set_name__(self, owner,
name)`, which stores `name` on the field. In a class `Product`, create
`title = Field()` and `price = Field()`. Print the names the two fields
recorded:

```text
title price
```

<details><summary><b>Hint</b></summary>

You never call `__set_name__` yourself. Python calls it for every object in the class body that has the method, once the class is built.

</details>

<details><summary><b>Expected output</b></summary>

```text
title price
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Field:
    def __set_name__(self, owner, name):
        self.name = name


class Product:
    title = Field()
    price = Field()


print(Product.title.name, Product.price.name)
```

When the class body finishes, `type` walks through the new namespace and calls `__set_name__(Product, "title")` on every value that defines it. That's how a descriptor learns its own attribute name without you typing it twice (`title = Field("title")`). `Field` has no `__get__`, so `Product.title` hands back the `Field` object itself.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Field:
    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name

    def __repr__(self):
        return f"Field({self.owner.__name__}.{self.name})"


class Product:
    title = Field()
    price = Field()


print(Product.title.name, Product.price.name)
```

Remembering the owner too, plus a `__repr__`, pays off in error messages: `Field(Product.price)` says exactly where a bad value went. Before Python 3.6 added `__set_name__`, frameworks needed a metaclass just to tell each field its name.

</details>

In [ ]:
# your code here


### NB25-Q06 · L2 · __getattr__ · Predict

When does `__getattr__` run? Predict the output.

**What does this print?**

```python
class Account:
    def __init__(self):
        self.balance = 100

    def __getattr__(self, name):
        print("getattr:", name)
        return 0


acct = Account()
print(acct.balance)
print(acct.bonus)
```

<details><summary><b>Hint</b></summary>

`__getattr__` is a fallback, not a gatekeeper.

</details>

<details><summary><b>Expected output</b></summary>

```text
100
getattr: bonus
0
```

</details>

<details><summary><b>Solution</b></summary>

`acct.balance` is found in the instance's `__dict__`, so `__getattr__` is never called and `100` prints. `acct.bonus` isn't found anywhere, so Python calls `__getattr__(acct, "bonus")`, which prints its message and returns `0`. The method that runs on *every* read is `__getattribute__` (Q12).

</details>

<details><summary><b>Senior dev solution</b></summary>

Because it only runs on misses, `__getattr__` is cheap and safe to add, which is why seniors use it (not `__getattribute__`) for proxies and fallbacks.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB25-Q07 · L2 · __setattr__ · Write

Write a class `Order` whose `__setattr__` prints `set <name> = <value>`
(with the value shown by `repr`) and then really stores the attribute.
Its `__init__` sets `status` to `"new"`. Create an order, set its
`status` to `"paid"`, then print the status:

```text
set status = 'new'
set status = 'paid'
paid
```

<details><summary><b>Hint</b></summary>

Inside `__setattr__`, writing `self.status = ...` would call `__setattr__` again. Hand the real storing to `object.__setattr__(self, name, value)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
set status = 'new'
set status = 'paid'
paid
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Order:
    def __init__(self):
        self.status = "new"

    def __setattr__(self, name, value):
        print(f"set {name} = {value!r}")
        object.__setattr__(self, name, value)


order = Order()
order.status = "paid"
print(order.status)
```

`__setattr__` runs on *every* assignment to an attribute, including the one inside `__init__`. That's why there are two `set` lines. `object.__setattr__` is the default behaviour that actually stores the value, so calling it avoids looping back into your own method.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Order:
    def __init__(self):
        self.status = "new"

    def __setattr__(self, name, value):
        print(f"set {name} = {value!r}")
        super().__setattr__(name, value)


order = Order()
order.status = "paid"
print(order.status)
```

`super().__setattr__` goes through any parent class that also customises setting (cooperative inheritance, NB21), where `object.__setattr__` jumps straight past it. Real code uses this hook for change tracking, read-only objects and validation.

</details>

In [ ]:
# your code here


### NB25-Q08 · L2 · Methods are descriptors · Predict

Methods and classmethods are descriptors too. Predict the output.

**What does this print?**

```python
class Dog:
    def speak(self):
        return "Woof"

    @classmethod
    def kind(cls):
        return cls.__name__


raw = vars(Dog)["speak"]
rex = Dog()
print(hasattr(raw, "__get__"), hasattr(raw, "__set__"))
print(raw.__get__(rex, Dog)())
print(Dog.speak is raw, rex.speak is raw)
print(vars(Dog)["kind"].__get__(None, Dog)())
```

<details><summary><b>Hint</b></summary>

A plain function has `__get__`. Reading it through an instance calls `__get__`, which builds a new *bound method*. Reading it through the class gives back the function itself.

</details>

<details><summary><b>Expected output</b></summary>

```text
True False
Woof
True False
Dog
```

</details>

<details><summary><b>Solution</b></summary>

Every function has a `__get__` method (and no `__set__`), so functions are *non-data descriptors*. `raw.__get__(rex, Dog)` is what `rex.speak` does: it returns a bound method that remembers `rex` as `self`, which is why calling it gives `Woof`. Reading through the class (`Dog.speak`) returns the plain function, so `is` is `True`; through an instance you get a new bound method each time, so `is` is `False`. A `classmethod` object's `__get__` binds the *class* instead, so the last line prints `Dog`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is the whole secret of `self`: there's no magic in the method, only in the lookup. `staticmethod`, `classmethod` and `property` are just three different `__get__` implementations.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB25-Q09 · L2 · inspect · Write

Use `inspect.getmembers` with `inspect.isfunction` to print, as a list, the
names of the *public* functions defined on `Calculator` (names that don't
start with `_`):

```text
['add', 'subtract', 'version']
```

**Starting code (already in the cell below):**

```python
import inspect


class Calculator:
    def __init__(self):
        self.memory = 0

    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b

    def _round(self, value):
        return round(value, 2)

    @staticmethod
    def version():
        return "1.0"
```

<details><summary><b>Hint</b></summary>

`inspect.getmembers(obj, predicate)` returns `(name, value)` pairs for every member where `predicate(value)` is true, sorted by name.

</details>

<details><summary><b>Expected output</b></summary>

```text
['add', 'subtract', 'version']
```

</details>

<details><summary><b>Solution</b></summary>

```python
public = []
for name, member in inspect.getmembers(Calculator, inspect.isfunction):
    if not name.startswith("_"):
        public.append(name)
print(public)
```

Read through the class, methods are plain functions (Q08), so `inspect.isfunction` matches them, including the `staticmethod`. `getmembers` already sorts by name. The loop then skips `__init__` and `_round` because they start with an underscore.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
public = [name for name, _ in inspect.getmembers(Calculator, inspect.isfunction)
          if not name.startswith("_")]
print(public)
```

A comprehension (NB10) with `_` for the unused value says "I only want the names". On an *instance*, methods come back as bound methods, so you'd need `inspect.ismethod` there. Knowing which one applies comes straight from Q08.

</details>

In [ ]:
import inspect


class Calculator:
    def __init__(self):
        self.memory = 0

    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b

    def _round(self, value):
        return round(value, 2)

    @staticmethod
    def version():
        return "1.0"

# your code here


---

## L3 · Intermediate

### NB25-Q10 · L3 · Descriptors · Write

Write a data descriptor `Typed` that takes a type when created
(`Typed(int)`). Use `__set_name__` to learn the attribute name.
`__set__` raises `TypeError` with the message
`<name> must be <type>, got <type>` when the value is the wrong type,
and stores it otherwise. `__get__` returns the stored value.

Use it in `Product` for `name` (`str`) and `quantity` (`int`). Create
`Product("bolt", 40)`, print its name and quantity, then try to set
`quantity` to `"50"` and print the error message:

```text
bolt 40
quantity must be int, got str
```

<details><summary><b>Hint</b></summary>

Store each value in the instance's own dict: `obj.__dict__[self.name] = value`. `type(value).__name__` gives the type's name as text.

</details>

<details><summary><b>Expected output</b></summary>

```text
bolt 40
quantity must be int, got str
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Typed:
    def __init__(self, kind):
        self.kind = kind

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if not isinstance(value, self.kind):
            raise TypeError(f"{self.name} must be {self.kind.__name__}, "
                            f"got {type(value).__name__}")
        obj.__dict__[self.name] = value


class Product:
    name = Typed(str)
    quantity = Typed(int)

    def __init__(self, name, quantity):
        self.name = name
        self.quantity = quantity


bolt = Product("bolt", 40)
print(bolt.name, bolt.quantity)
try:
    bolt.quantity = "50"
except TypeError as error:
    print(error)
```

`Typed` defines `__set__`, which makes it a *data descriptor*, and data descriptors take priority over the instance `__dict__` (Q11). So storing the value in `obj.__dict__` under the same name is safe: every read and write still goes through the descriptor. `__init__` is checked too, because `self.quantity = quantity` is an ordinary assignment.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Typed:
    def __init__(self, kind):
        self.kind = kind

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if not isinstance(value, self.kind):
            raise TypeError(f"{self.name} must be {self.kind.__name__}, "
                            f"got {type(value).__name__}")
        obj.__dict__[self.name] = value


class Product:
    name = Typed(str)
    quantity = Typed(int)

    def __init__(self, name, quantity):
        self.name = name
        self.quantity = quantity


bolt = Product("bolt", 40)
print(bolt.name, bolt.quantity)
try:
    bolt.quantity = "50"
except TypeError as error:
    print(error)
```

The `if obj is None: return self` guard makes `Product.quantity` (read from the class) return the descriptor instead of crashing, so `help()`, `inspect` and your own tooling can see it. One more edge case: `isinstance(True, int)` is `True`, so `Typed(int)` accepts booleans unless you reject them on purpose.

</details>

In [ ]:
# your code here


### NB25-Q11 · L3 · Data vs non-data descriptors · Predict

Two descriptors and an instance dict that disagree. Predict the output.

**What does this print?**

```python
class NonData:
    def __get__(self, obj, objtype=None):
        return "from descriptor"


class Data:
    def __get__(self, obj, objtype=None):
        return "from descriptor"

    def __set__(self, obj, value):
        pass


class Item:
    a = NonData()
    b = Data()


item = Item()
item.__dict__["a"] = "from instance"
item.__dict__["b"] = "from instance"
print(item.a)
print(item.b)
```

<details><summary><b>Hint</b></summary>

The only difference between the two descriptors is `__set__`. It decides who wins against the instance's own dict.

</details>

<details><summary><b>Expected output</b></summary>

```text
from instance
from descriptor
```

</details>

<details><summary><b>Solution</b></summary>

Attribute lookup goes in this order: (1) a *data* descriptor on the class (one with `__set__` or `__delete__`), (2) the instance `__dict__`, (3) a *non-data* descriptor or plain class attribute. So the instance value hides `a`, but `b`'s data descriptor wins over the instance value.

</details>

<details><summary><b>Senior dev solution</b></summary>

This order is deliberate. Methods are non-data descriptors, so you can override one on a single instance. Properties are data descriptors, so an instance can't accidentally hide them. `functools.cached_property` (Q30) relies on being non-data.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB25-Q12 · L3 · __getattribute__ · Error

This class tries to customise every attribute read. Which error does it raise?

**Which error does this raise?**

```python
class Account:
    def __init__(self, owner):
        self.owner = owner

    def __getattribute__(self, name):
        return self.__dict__[name]


acct = Account("Ada")
print(acct.owner)
```

<details><summary><b>Hint</b></summary>

`__getattribute__` runs for *every* attribute read on the object. Is `self.__dict__` an attribute read?

</details>

<details><summary><b>Expected output</b></summary>

```text
RecursionError
```

</details>

<details><summary><b>Solution</b></summary>

`RecursionError`. Reading `self.__dict__` is itself an attribute read, so it calls `__getattribute__(self, "__dict__")`, which reads `self.__dict__` again, forever, until Python gives up. Inside `__getattribute__`, every read of `self` must go through `object.__getattribute__(self, name)`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Account:
    def __init__(self, owner):
        self.owner = owner

    def __getattribute__(self, name):
        return object.__getattribute__(self, name)


acct = Account("Ada")
print(acct.owner)
```

Seniors almost never override `__getattribute__`: it slows down every read, and one slip recurses. If the goal is a fallback for missing names, `__getattr__` (Q06) is the safe tool.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB25-Q13 · L3 · __setattr__ · Fix

The thermostat should clamp its target between 10 and 30, so this should
print `30`. It crashes instead. Fix it.

**Buggy code (copied into the cell below):**

```python
class Thermostat:
    def __init__(self, target):
        self.target = target

    def __setattr__(self, name, value):
        if name == "target":
            value = max(10, min(value, 30))
        setattr(self, name, value)


thermostat = Thermostat(35)
print(thermostat.target)
```

<details><summary><b>Hint</b></summary>

`setattr(self, name, value)` is the same as `self.<name> = value`. What does that call?

</details>

<details><summary><b>Expected output</b></summary>

```text
30
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Thermostat:
    def __init__(self, target):
        self.target = target

    def __setattr__(self, name, value):
        if name == "target":
            value = max(10, min(value, 30))
        object.__setattr__(self, name, value)


thermostat = Thermostat(35)
print(thermostat.target)
```

`setattr(self, name, value)` triggers `__setattr__` again, which calls `setattr` again, until `RecursionError`. `object.__setattr__` is the built-in storing step, so it ends the chain.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Thermostat:
    def __init__(self, target):
        self.target = target

    @property
    def target(self):
        return self._target

    @target.setter
    def target(self, value):
        self._target = max(10, min(value, 30))


thermostat = Thermostat(35)
print(thermostat.target)
```

A `__setattr__` that only cares about one attribute runs for *every* attribute. A property (NB21) targets exactly the one that needs rules, and can't recurse by accident as long as it stores under a different name (`_target`).

</details>

In [ ]:
class Thermostat:
    def __init__(self, target):
        self.target = target

    def __setattr__(self, name, value):
        if name == "target":
            value = max(10, min(value, 30))
        setattr(self, name, value)


thermostat = Thermostat(35)
print(thermostat.target)


### NB25-Q14 · L3 · Monkey-patching · Write

`report()` needs the network, which this notebook doesn't have. Replace
`WeatherService.fetch_temperature` with a fake that returns `18`, print
`report(WeatherService(), "Oslo")`, and put the original method back
**even if something goes wrong**. Finally, print whether the class holds
the original again (compare with `is`):

```text
Oslo: 18 C
True
```

**Starting code (already in the cell below):**

```python
class WeatherService:
    def fetch_temperature(self, city):
        raise ConnectionError("no network in this notebook")


def report(service, city):
    return f"{city}: {service.fetch_temperature(city)} C"
```

<details><summary><b>Hint</b></summary>

Save the original first. Assign your fake function to the *class* attribute. `try` / `finally` guarantees the restore runs.

</details>

<details><summary><b>Expected output</b></summary>

```text
Oslo: 18 C
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
original = WeatherService.fetch_temperature


def fake_fetch(self, city):
    return 18


WeatherService.fetch_temperature = fake_fetch
try:
    print(report(WeatherService(), "Oslo"))
finally:
    WeatherService.fetch_temperature = original
print(WeatherService.fetch_temperature is original)
```

Classes are ordinary objects you can change at runtime: that's *monkey-patching*. Because the fake is assigned to the class, every instance picks it up and it's bound like any method, so it needs `self`. The `finally` block restores the original whether or not `report` raises, so the patch can't leak into later code.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from unittest import mock

original = WeatherService.fetch_temperature
with mock.patch.object(WeatherService, "fetch_temperature", return_value=18):
    print(report(WeatherService(), "Oslo"))
print(WeatherService.fetch_temperature is original)
```

`mock.patch.object` (NB28) does the save, replace and guaranteed restore in one `with` line, and the fake records how it was called. Seniors keep monkey-patching for tests. Patching a library in production code makes bugs that nobody can find by reading the source.

</details>

In [ ]:
class WeatherService:
    def fetch_temperature(self, city):
        raise ConnectionError("no network in this notebook")


def report(service, city):
    return f"{city}: {service.fetch_temperature(city)} C"

# your code here


### NB25-Q15 · L3 · Monkey-patching · Fix

We want to give every `Greeter` a `shout` method at runtime, so that
`greeter.shout()` prints `ADA!`. It crashes instead. Fix it.

**Buggy code (copied into the cell below):**

```python
class Greeter:
    def __init__(self, name):
        self.name = name


def shout(self):
    return self.name.upper() + "!"


greeter = Greeter("ada")
greeter.shout = shout
print(greeter.shout())
```

<details><summary><b>Hint</b></summary>

A function only turns into a method (with `self` filled in) when it's found on the *class*. Where was it stored here?

</details>

<details><summary><b>Expected output</b></summary>

```text
ADA!
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Greeter:
    def __init__(self, name):
        self.name = name


def shout(self):
    return self.name.upper() + "!"


greeter = Greeter("ada")
Greeter.shout = shout
print(greeter.shout())
```

`greeter.shout = shout` puts the plain function in the *instance* dict. Lookup finds it there and returns it as-is: no `__get__`, no binding, so `shout()` is called with no `self` and raises `TypeError`. Assigning to the class means lookup goes through the function's `__get__` (Q08), which binds `self` for every instance.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import types


class Greeter:
    def __init__(self, name):
        self.name = name


def shout(self):
    return self.name.upper() + "!"


greeter = Greeter("ada")
greeter.shout = types.MethodType(shout, greeter)
print(greeter.shout())
```

If you really want to change only *one* object, `types.MethodType(func, obj)` binds the function by hand, which is exactly what `__get__` does for you. Patching the class changes every instance, including ones other code already holds, so seniors choose deliberately.

</details>

In [ ]:
class Greeter:
    def __init__(self, name):
        self.name = name


def shout(self):
    return self.name.upper() + "!"


greeter = Greeter("ada")
greeter.shout = shout
print(greeter.shout())


### NB25-Q16 · L3 · vars() and dir() · Predict

`vars()` versus `dir()` with inheritance. Predict the output.

**What does this print?**

```python
class Vehicle:
    wheels = 4

    def honk(self):
        return "beep"


class Van(Vehicle):
    def __init__(self, plate):
        self.plate = plate


van = Van("AB-123")
print(vars(van))
print("wheels" in dir(van), "wheels" in vars(van))
print([name for name in dir(van) if not name.startswith("_")])
```

<details><summary><b>Hint</b></summary>

`vars()` is one dict. `dir()` collects names from the object, its class and every parent class, and sorts them.

</details>

<details><summary><b>Expected output</b></summary>

```text
{'plate': 'AB-123'}
True False
['honk', 'plate', 'wheels']
```

</details>

<details><summary><b>Solution</b></summary>

`vars(van)` is only what's stored on the instance: `{'plate': 'AB-123'}`. `dir(van)` gathers names from the instance, `Van`, `Vehicle` and `object`, and returns them sorted. So `wheels` is in `dir` but not in `vars`, and the public names are `honk`, `plate` and `wheels` in alphabetical order.

</details>

<details><summary><b>Senior dev solution</b></summary>

Use `vars()` to see *state* and `dir()` to see *what you can call*. `dir()` is meant for exploring, not for program logic: a class can customise it with `__dir__`, so it isn't a complete or exact list.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB25-Q17 · L3 · eval and exec · Write

Config text arrives from a file. Parse `settings_text` with
`ast.literal_eval` and print the `retries` value and the number of hosts.
Then try to parse `attack_text` the same way. When it's refused with
`ValueError`, print `refused`:

```text
3 2
refused
```

**Starting code (already in the cell below):**

```python
import ast

settings_text = "{'retries': 3, 'hosts': ['a.local', 'b.local']}"
attack_text = "__import__('os').getcwd()"
```

<details><summary><b>Hint</b></summary>

`ast.literal_eval` turns text into a value, but only for literals: numbers, strings, lists, dicts, tuples, sets, booleans and `None`.

</details>

<details><summary><b>Expected output</b></summary>

```text
3 2
refused
```

</details>

<details><summary><b>Solution</b></summary>

```python
settings = ast.literal_eval(settings_text)
print(settings["retries"], len(settings["hosts"]))
try:
    ast.literal_eval(attack_text)
except ValueError:
    print("refused")
```

`eval(text)` would *run* the text as code, and `attack_text` imports `os`, which could just as easily delete files. `ast.literal_eval` parses the text and accepts only literal values, so a function call is refused with `ValueError` before anything runs.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def parse_literal(text):
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None


settings = parse_literal(settings_text)
print(settings["retries"], len(settings["hosts"]))
print("refused" if parse_literal(attack_text) is None else "accepted")
```

`literal_eval` also raises `SyntaxError` for text that isn't Python at all, so a helper that catches both gives one clear "refused" path. Never pass untrusted text to `eval` or `exec`, not even with an emptied `__builtins__`: people have escaped every such sandbox. For config files, seniors prefer a real data format such as JSON (NB13) or TOML (NB31).

</details>

In [ ]:
import ast

settings_text = "{'retries': 3, 'hosts': ['a.local', 'b.local']}"
attack_text = "__import__('os').getcwd()"

# your code here


### NB25-Q18 · L3 · __class_getitem__ · Write

ORMs often let you write `Column[int]`. Give a class `Column` an
`__init__(self, kind)` that stores `kind`, and a
`__class_getitem__(cls, kind)` that returns `cls(kind)`. Print the type
names stored by `Column[int]` and `Column[str]`, and whether `Column[int]`
is a `Column` instance:

```text
int str True
```

<details><summary><b>Hint</b></summary>

Square brackets on a *class* call `__class_getitem__`. It's automatically a classmethod, so you don't need the `@classmethod` decorator.

</details>

<details><summary><b>Expected output</b></summary>

```text
int str True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Column:
    def __init__(self, kind):
        self.kind = kind

    def __class_getitem__(cls, kind):
        return cls(kind)


print(Column[int].kind.__name__, Column[str].kind.__name__,
      isinstance(Column[int], Column))
```

`Column[int]` becomes `Column.__class_getitem__(int)`. Python passes the class as `cls` without you writing `@classmethod`, and here it builds an instance. (`obj[key]` on an *instance* calls `__getitem__` instead.)

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Column:
    def __init__(self, kind):
        self.kind = kind

    def __class_getitem__(cls, kind):
        return cls(kind)


print(Column[int].kind.__name__, Column[str].kind.__name__,
      isinstance(Column[int], Column))
```

Same code. Python's own generics use this hook: `list[int]` returns a `types.GenericAlias`, which exists only for type hints (NB26). Using it to build real objects is fine in your own library, but document it, because readers expect `X[...]` to mean "a type hint".

</details>

In [ ]:
# your code here


### NB25-Q19 · L3 · Descriptors · Refactor

Three copy-pasted properties do the same check. Replace them with one
reusable descriptor, `Positive`, without changing the output.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
class Shipment:
    def __init__(self, weight, length, width):
        self.weight = weight
        self.length = length
        self.width = width

    @property
    def weight(self):
        return self._weight

    @weight.setter
    def weight(self, value):
        if value <= 0:
            raise ValueError("weight must be positive")
        self._weight = value

    @property
    def length(self):
        return self._length

    @length.setter
    def length(self, value):
        if value <= 0:
            raise ValueError("length must be positive")
        self._length = value

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        if value <= 0:
            raise ValueError("width must be positive")
        self._width = value


box = Shipment(2.5, 40, 30)
print(box.weight, box.length, box.width)
try:
    box.width = 0
except ValueError as error:
    print(error)
```

<details><summary><b>Hint</b></summary>

One descriptor class with `__set_name__` (to know which attribute it guards), `__get__` and `__set__`. Then each attribute becomes one line: `weight = Positive()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
2.5 40 30
width must be positive
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Positive:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")
        obj.__dict__[self.name] = value


class Shipment:
    weight = Positive()
    length = Positive()
    width = Positive()

    def __init__(self, weight, length, width):
        self.weight = weight
        self.length = length
        self.width = width


box = Shipment(2.5, 40, 30)
print(box.weight, box.length, box.width)
try:
    box.width = 0
except ValueError as error:
    print(error)
```

Each property was a hand-written descriptor for one attribute. `Positive` is the same logic written once; `__set_name__` tells each copy whether it's guarding `weight`, `length` or `width`, so the error message stays correct. Thirty lines of properties become three lines of declarations.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Positive:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")
        obj.__dict__[self.name] = value


class Shipment:
    weight = Positive()
    length = Positive()
    width = Positive()

    def __init__(self, weight, length, width):
        self.weight = weight
        self.length = length
        self.width = width


box = Shipment(2.5, 40, 30)
print(box.weight, box.length, box.width)
try:
    box.width = 0
except ValueError as error:
    print(error)
```

The rule now lives in one place, and any class can reuse it (`Order.quantity = Positive()`). The class-access guard keeps `Shipment.weight` usable for introspection. The rule of three applies: one property is fine, three identical ones are a descriptor waiting to happen.

</details>

In [ ]:
class Shipment:
    def __init__(self, weight, length, width):
        self.weight = weight
        self.length = length
        self.width = width

    @property
    def weight(self):
        return self._weight

    @weight.setter
    def weight(self, value):
        if value <= 0:
            raise ValueError("weight must be positive")
        self._weight = value

    @property
    def length(self):
        return self._length

    @length.setter
    def length(self, value):
        if value <= 0:
            raise ValueError("length must be positive")
        self._length = value

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        if value <= 0:
            raise ValueError("width must be positive")
        self._width = value


box = Shipment(2.5, 40, 30)
print(box.weight, box.length, box.width)
try:
    box.width = 0
except ValueError as error:
    print(error)


---

## L4 · Somewhat Difficult

### NB25-Q20 · L4 · __init_subclass__ · Write

Build a plugin registry for exporters. Base class `Exporter` has a class
attribute `registry = {}` and an `__init_subclass__(cls, fmt, **kwargs)`
that calls `super().__init_subclass__(**kwargs)` and stores
`registry[fmt] = cls`.

Define `CsvExporter(Exporter, fmt="csv")` and
`JsonExporter(Exporter, fmt="json")`. Each has an `export(rows)` method
returning `"<fmt>: <n> rows"`. Print the sorted registry keys, then look
up each format by name and export `rows`:

```text
['csv', 'json']
csv: 3 rows
json: 3 rows
```

**Starting code (already in the cell below):**

```python
rows = [{"id": 1}, {"id": 2}, {"id": 3}]
```

<details><summary><b>Hint</b></summary>

Keyword arguments in the `class` line (like `fmt="csv"`) are passed to the parent's `__init_subclass__`. It runs once, when the subclass is defined.

</details>

<details><summary><b>Expected output</b></summary>

```text
['csv', 'json']
csv: 3 rows
json: 3 rows
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Exporter:
    registry = {}

    def __init_subclass__(cls, fmt, **kwargs):
        super().__init_subclass__(**kwargs)
        Exporter.registry[fmt] = cls


class CsvExporter(Exporter, fmt="csv"):
    def export(self, rows):
        return f"csv: {len(rows)} rows"


class JsonExporter(Exporter, fmt="json"):
    def export(self, rows):
        return f"json: {len(rows)} rows"


print(sorted(Exporter.registry))
for fmt in ["csv", "json"]:
    exporter = Exporter.registry[fmt]()
    print(exporter.export(rows))
```

`__init_subclass__` is a hook on the *parent* that runs each time a subclass is created. The class keyword `fmt="csv"` is passed straight to it. Defining the class is enough to register it, so a new exporter needs no change anywhere else. Python makes it a classmethod automatically.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Exporter:
    registry = {}

    def __init_subclass__(cls, fmt, **kwargs):
        super().__init_subclass__(**kwargs)
        if fmt in Exporter.registry:
            raise TypeError(f"format {fmt!r} is already registered")
        cls.fmt = fmt
        Exporter.registry[fmt] = cls

    def export(self, rows):
        return f"{self.fmt}: {len(rows)} rows"


class CsvExporter(Exporter, fmt="csv"):
    pass


class JsonExporter(Exporter, fmt="json"):
    pass


print(sorted(Exporter.registry))
for fmt in ["csv", "json"]:
    print(Exporter.registry[fmt]().export(rows))
```

The duplicate check turns a copy-paste mistake (two classes claiming `csv`) into an error at import time instead of a silent overwrite. Storing `fmt` on the class lets shared behaviour live in the base. Since Python 3.6 this hook replaces most registry metaclasses.

</details>

In [ ]:
rows = [{"id": 1}, {"id": 2}, {"id": 3}]

# your code here


### NB25-Q21 · L4 · Metaclasses · Predict

When does a metaclass run? Predict the output.

**What does this print?**

```python
class Meta(type):
    def __new__(mcls, name, bases, namespace):
        public = sorted(key for key in namespace if not key.startswith("__"))
        print("Meta.__new__:", name, public)
        return super().__new__(mcls, name, bases, namespace)


print("before")


class Report(metaclass=Meta):
    print("class body runs")
    title = "Q3"

    def render(self):
        return self.title


print("after")
report = Report()
print(report.render(), type(Report).__name__)
```

<details><summary><b>Hint</b></summary>

A class body is ordinary code that runs top to bottom first. Only then is the finished namespace handed to the metaclass. Making an *instance* is a different step.

</details>

<details><summary><b>Expected output</b></summary>

```text
before
class body runs
Meta.__new__: Report ['render', 'title']
after
Q3 Meta
```

</details>

<details><summary><b>Solution</b></summary>

The class body runs first (printing `class body runs`) and fills a namespace dict. Then Python calls `Meta.__new__` with that dict to build the class, so it sees `render` and `title` (plus dunders like `__module__`, which we filtered out). Creating `Report()` later doesn't call `Meta.__new__` again: that only builds *classes*. And the class of `Report` is `Meta`, just as the class of a normal class is `type`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Everything a metaclass does happens at class-definition time, usually at import. That makes metaclass bugs show up as import errors far from where the class is used, one reason seniors prefer lighter tools.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB25-Q22 · L4 · Metaclasses · Write

Write a metaclass `Singleton` whose `__call__` creates an instance only
the first time a class is called, and returns that same instance after
that. Keep the instances in a dict on the metaclass, keyed by class.

Use it for `AppConfig`, whose `__init__` prints `loading config`. Call
`AppConfig()` twice and print whether both results are the same object:

```text
loading config
True
```

<details><summary><b>Hint</b></summary>

`AppConfig()` really runs `type(AppConfig).__call__(AppConfig)`, which normally calls `__new__` and `__init__`. Override `__call__` and only call `super().__call__(*args, **kwargs)` when there's no saved instance yet.

</details>

<details><summary><b>Expected output</b></summary>

```text
loading config
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Singleton(type):
    instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in Singleton.instances:
            Singleton.instances[cls] = super().__call__(*args, **kwargs)
        return Singleton.instances[cls]


class AppConfig(metaclass=Singleton):
    def __init__(self):
        print("loading config")


first = AppConfig()
second = AppConfig()
print(first is second)
```

Calling a class is calling an object, so Python looks for `__call__` on the object's class: the metaclass. The default `type.__call__` creates and initialises a new instance. Ours does that once, saves the result, and returns the saved one every later time, so `__init__` runs only once.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import functools


class AppConfig:
    def __init__(self):
        print("loading config")


@functools.cache
def get_config():
    return AppConfig()


first = get_config()
second = get_config()
print(first is second)
```

A singleton metaclass hides the "only one" rule inside what looks like a normal `AppConfig()` call, which surprises readers and makes tests painful (you can never get a fresh one). A cached factory function (NB11) or a module-level instance says plainly what it does.

</details>

In [ ]:
# your code here


### NB25-Q23 · L4 · Descriptors · Fix

Alice opened her account with 100 and Bob with 5, but both accounts show
`5`. Fix the descriptor so each account keeps its own balance:

```text
Alice 100
Bob 5
```

**Buggy code (copied into the cell below):**

```python
class NonNegative:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        return self.value

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError(f"{self.name} cannot be negative")
        self.value = value


class Account:
    balance = NonNegative()

    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance


alice = Account("Alice", 100)
bob = Account("Bob", 5)
print(alice.owner, alice.balance)
print(bob.owner, bob.balance)
```

<details><summary><b>Hint</b></summary>

How many `NonNegative` objects exist? One per *class attribute*, not one per account. So where should each account's value live?

</details>

<details><summary><b>Expected output</b></summary>

```text
Alice 100
Bob 5
```

</details>

<details><summary><b>Solution</b></summary>

```python
class NonNegative:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError(f"{self.name} cannot be negative")
        obj.__dict__[self.name] = value


class Account:
    balance = NonNegative()

    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance


alice = Account("Alice", 100)
bob = Account("Bob", 5)
print(alice.owner, alice.balance)
print(bob.owner, bob.balance)
```

The descriptor is created once, in the class body, and shared by every `Account`. Storing on `self.value` means Bob's deposit overwrote Alice's. The `obj` argument is the specific account, so storing in `obj.__dict__` gives every instance its own value.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class NonNegative:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError(f"{self.name} cannot be negative")
        obj.__dict__[self.name] = value


class Account:
    balance = NonNegative()

    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance


alice = Account("Alice", 100)
bob = Account("Bob", 5)
print(alice.owner, alice.balance)
print(bob.owner, bob.balance)
```

This is the most common descriptor bug, and it only shows up with two or more instances, so test with two. For classes without a `__dict__` (`__slots__`), seniors key a `weakref.WeakKeyDictionary` (NB29) by instance instead.

</details>

In [ ]:
class NonNegative:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        return self.value

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError(f"{self.name} cannot be negative")
        self.value = value


class Account:
    balance = NonNegative()

    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance


alice = Account("Alice", 100)
bob = Account("Bob", 5)
print(alice.owner, alice.balance)
print(bob.owner, bob.balance)


### NB25-Q24 · L4 · property is a descriptor · Write

`property` is a class you could write yourself. Write `my_property`:

- `__init__(self, fget, fset=None)` stores the getter and setter.
- `__get__` returns the descriptor itself when `obj` is `None`, otherwise
  `self.fget(obj)`.
- `__set__` calls `self.fset(obj, value)`, or raises
  `AttributeError("read-only")` when there is no setter.
- `setter(self, fset)` returns a **new** `my_property` with both functions.

Use it in `Circle`: `radius` (getter and setter, stored as `_radius`) and
`diameter` (getter only). Create `Circle(2)`, print radius and diameter,
set radius to `5`, print both again, then try setting `diameter` to `3`
and print the error:

```text
2 4
5 10
read-only
```

<details><summary><b>Hint</b></summary>

`@my_property` above `def radius` calls `my_property(radius_function)`. `@radius.setter` above the second `def radius` calls the *first* object's `setter` method, and its return value replaces the name `radius`.

</details>

<details><summary><b>Expected output</b></summary>

```text
2 4
5 10
read-only
```

</details>

<details><summary><b>Solution</b></summary>

```python
class my_property:
    def __init__(self, fget, fset=None):
        self.fget = fget
        self.fset = fset

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("read-only")
        self.fset(obj, value)

    def setter(self, fset):
        return my_property(self.fget, fset)


class Circle:
    def __init__(self, radius):
        self.radius = radius

    @my_property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        self._radius = value

    @my_property
    def diameter(self):
        return self._radius * 2


circle = Circle(2)
print(circle.radius, circle.diameter)
circle.radius = 5
print(circle.radius, circle.diameter)
try:
    circle.diameter = 3
except AttributeError as error:
    print(error)
```

`my_property` defines `__set__`, so it's a data descriptor: it wins over the instance dict and every read or write of `radius` goes through it. `setter` returns a new object holding both functions, and because the setter is also named `radius`, that new object replaces the old one in the class body. `diameter` has no setter, and since `__set__` still exists, assignment is blocked rather than quietly creating an instance attribute.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        self._radius = value

    @property
    def diameter(self):
        return self._radius * 2


circle = Circle(2)
print(circle.radius, circle.diameter)
circle.radius = 5
print(circle.radius, circle.diameter)
try:
    circle.diameter = 3
except AttributeError:
    print("read-only")
```

The real `property` is written in C but follows this exact design, which explains its rules: the setter must reuse the property's name, and a property without a setter is read-only rather than overridable. Its own error message differs by version, so the senior prints a fixed text.

</details>

In [ ]:
# your code here


### NB25-Q25 · L4 · ABC.register and __subclasshook__ · Write

Write an abstract base class `Drawable` (from `abc.ABC`) with a
`__subclasshook__` classmethod: if the class being checked has a `draw`
attribute, return `True`; otherwise return `NotImplemented` (**not**
`False`). Then:

1. Print `isinstance` checks against `Drawable` for a `Circle()` and a
   `Label()` instance.
2. Call `Drawable.register(Label)` and print the `Label()` check again.

```text
True False
True
```

**Starting code (already in the cell below):**

```python
from abc import ABC


class Circle:
    def draw(self):
        return "()"


class Label:
    def render(self):
        return "text"
```

<details><summary><b>Hint</b></summary>

`__subclasshook__(cls, other)` decides `issubclass(other, Drawable)`, and `isinstance` asks it too. Returning `NotImplemented` means "I don't know, use the normal rules", which include registered classes.

</details>

<details><summary><b>Expected output</b></summary>

```text
True False
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Drawable(ABC):
    @classmethod
    def __subclasshook__(cls, other):
        if hasattr(other, "draw"):
            return True
        return NotImplemented


print(isinstance(Circle(), Drawable), isinstance(Label(), Drawable))
Drawable.register(Label)
print(isinstance(Label(), Drawable))
```

`Circle` never inherits from `Drawable`, but the hook says yes because it has `draw`: this is how `collections.abc.Iterable` recognises any class with `__iter__`. `Label` has no `draw`, so the hook returns `NotImplemented` and the normal rules answer `False`. `register` adds `Label` as a *virtual subclass*, and the normal rules now say `True`. Had the hook returned `False`, that answer would be final and `register` would have no effect.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Drawable(ABC):
    @classmethod
    def __subclasshook__(cls, other):
        if cls is Drawable and any("draw" in vars(klass) for klass in other.__mro__):
            return True
        return NotImplemented


print(isinstance(Circle(), Drawable), isinstance(Label(), Drawable))
Drawable.register(Label)
print(isinstance(Label(), Drawable))
```

Two habits from the standard library's own ABCs: `cls is Drawable` stops subclasses of `Drawable` inheriting a hook that's too loose, and checking the class dicts along the MRO ignores instance attributes. For new code, a `typing.Protocol` (NB26) states "anything with `draw()`" more clearly.

</details>

In [ ]:
from abc import ABC


class Circle:
    def draw(self):
        return "()"


class Label:
    def render(self):
        return "text"

# your code here


### NB25-Q26 · L4 · __instancecheck__ · Write

Make `isinstance(x, EvenNumber)` answer `True` exactly for even `int`
values. Write a metaclass `EvenMeta` with an `__instancecheck__(cls, obj)`
method, and a class `EvenNumber` that uses it. Print the checks for `4`,
`7`, `4.0` and `"8"`:

```text
True False False False
```

<details><summary><b>Hint</b></summary>

`isinstance(obj, C)` asks `type(C).__instancecheck__(C, obj)`. So the method belongs on the *metaclass*, not on `EvenNumber`.

</details>

<details><summary><b>Expected output</b></summary>

```text
True False False False
```

</details>

<details><summary><b>Solution</b></summary>

```python
class EvenMeta(type):
    def __instancecheck__(cls, obj):
        return type(obj) is int and obj % 2 == 0


class EvenNumber(metaclass=EvenMeta):
    pass


print(isinstance(4, EvenNumber), isinstance(7, EvenNumber),
      isinstance(4.0, EvenNumber), isinstance("8", EvenNumber))
```

`isinstance` is an ordinary function that asks the class's metaclass. Our `__instancecheck__` ignores inheritance completely and answers its own question. `type(obj) is int` rejects `4.0` and `"8"`, and also `True`, which `isinstance(True, int)` would let through.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def is_even_int(value):
    return type(value) is int and value % 2 == 0


print(is_even_int(4), is_even_int(7), is_even_int(4.0), is_even_int("8"))
```

Customising `isinstance` is clever, and clever is a cost: readers assume `isinstance` means inheritance. A plain function says what it checks. Use the hook only when an API demands a *class*, for example a `case EvenNumber():` pattern (NB18), which calls `isinstance` for you.

</details>

In [ ]:
# your code here


### NB25-Q27 · L4 · __init_subclass__ · Refactor

This metaclass only validates and announces subclasses, which
`__init_subclass__` can do without a metaclass. Refactor it, keeping the
output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
class TableMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if bases:
            if "table" not in namespace:
                raise TypeError(f"{name} needs a table name")
            print("registered", name, "->", namespace["table"])
        return cls


class Model(metaclass=TableMeta):
    pass


class User(Model):
    table = "users"


class Invoice(Model):
    table = "invoices"


try:
    class Broken(Model):
        pass
except TypeError as error:
    print(error)
```

<details><summary><b>Hint</b></summary>

`__init_subclass__` runs for every subclass but not for the base class itself, so the `if bases:` check disappears. Use `cls.__dict__` to see only what the subclass itself defined.

</details>

<details><summary><b>Expected output</b></summary>

```text
registered User -> users
registered Invoice -> invoices
Broken needs a table name
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Model:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if "table" not in cls.__dict__:
            raise TypeError(f"{cls.__name__} needs a table name")
        print("registered", cls.__name__, "->", cls.table)


class User(Model):
    table = "users"


class Invoice(Model):
    table = "invoices"


try:
    class Broken(Model):
        pass
except TypeError as error:
    print(error)
```

The metaclass ran after each class was built and checked its namespace. `__init_subclass__` runs at the same moment for every subclass, with the finished class as `cls`. `cls.__dict__` holds only what that subclass defined, matching the old `namespace` check (`hasattr` would wrongly accept an inherited `table`). No custom metaclass means no metaclass conflicts (Q32).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Model:
    def __init_subclass__(cls, table=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if table is None:
            raise TypeError(f"{cls.__name__} needs a table name")
        cls.table = table
        print("registered", cls.__name__, "->", table)


class User(Model, table="users"):
    pass


class Invoice(Model, table="invoices"):
    pass


try:
    class Broken(Model):
        pass
except TypeError as error:
    print(error)
```

Passing the table as a class keyword makes it impossible to inherit by accident and puts the setting right in the class line. The senior rule: reach for a metaclass only when you must change *how the class itself behaves* (like `__call__` or `__instancecheck__`), not to run code when a subclass appears.

</details>

In [ ]:
class TableMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if bases:
            if "table" not in namespace:
                raise TypeError(f"{name} needs a table name")
            print("registered", name, "->", namespace["table"])
        return cls


class Model(metaclass=TableMeta):
    pass


class User(Model):
    table = "users"


class Invoice(Model):
    table = "invoices"


try:
    class Broken(Model):
        pass
except TypeError as error:
    print(error)


---

## L5 · Hard

### NB25-Q28 · L5 · Metaclasses · Write

Write a metaclass `RequireAttrs`. When it builds a class, it reads the
class's `required` tuple and raises
`TypeError("<Class> is missing: <names>")` (names joined with `", "`, in
the order listed) for every name that isn't available on the new class,
**including through inheritance**. The base class that defines
`required` itself is exempt.

Base: `Sensor(metaclass=RequireAttrs)` with `required = ("unit", "read")`.
Define, each inside `try`/`except TypeError`:

- `Thermometer(Sensor)`: `unit = "C"` and a `read(self)` returning `21.5`
- `DigitalThermometer(Thermometer)`: an empty body
- `Barometer(Sensor)`: only `unit = "hPa"`
- `Hygrometer(Sensor)`: an empty body

For each class that's created, print `<Class> ok: <unit> <reading>`; for
each failure print the error:

```text
Thermometer ok: C 21.5
DigitalThermometer ok: C 21.5
Barometer is missing: read
Hygrometer is missing: unit, read
```

<details><summary><b>Hint</b></summary>

Build the class first with `super().__new__(...)`, then test the *class* with `hasattr`. Checking only `namespace` would wrongly reject `DigitalThermometer`, which inherits everything.

</details>

<details><summary><b>Expected output</b></summary>

```text
Thermometer ok: C 21.5
DigitalThermometer ok: C 21.5
Barometer is missing: read
Hygrometer is missing: unit, read
```

</details>

<details><summary><b>Solution</b></summary>

```python
class RequireAttrs(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if "required" not in namespace:
            missing = []
            for attr in cls.required:
                if not hasattr(cls, attr):
                    missing.append(attr)
            if missing:
                raise TypeError(f"{name} is missing: {', '.join(missing)}")
        return cls


class Sensor(metaclass=RequireAttrs):
    required = ("unit", "read")


try:
    class Thermometer(Sensor):
        unit = "C"

        def read(self):
            return 21.5
    print("Thermometer ok:", Thermometer.unit, Thermometer().read())
except TypeError as error:
    print(error)

try:
    class DigitalThermometer(Thermometer):
        pass
    print("DigitalThermometer ok:", DigitalThermometer.unit, DigitalThermometer().read())
except TypeError as error:
    print(error)

try:
    class Barometer(Sensor):
        unit = "hPa"
    print("Barometer ok:", Barometer.unit, Barometer().read())
except TypeError as error:
    print(error)

try:
    class Hygrometer(Sensor):
        pass
    print("Hygrometer ok:", Hygrometer.unit, Hygrometer().read())
except TypeError as error:
    print(error)
```

Every class in the family is built by `RequireAttrs`, because a metaclass is inherited along with the base. After building the class, `hasattr(cls, attr)` searches the whole MRO, so `DigitalThermometer` passes on what it inherited. The trap is checking `namespace`, which holds only what that one class body defined. `Sensor` is skipped because it defines `required` itself. Raising inside `__new__` means the failing class name is never created.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Sensor:
    required = ("unit", "read")

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        missing = [attr for attr in cls.required if not hasattr(cls, attr)]
        if missing:
            raise TypeError(f"{cls.__name__} is missing: {', '.join(missing)}")


def try_build(name, bases, namespace):
    try:
        cls = type(name, bases, namespace)
    except TypeError as error:
        print(error)
        return None
    print(f"{name} ok:", cls.unit, cls().read())
    return cls


Thermometer = try_build("Thermometer", (Sensor,), {"unit": "C", "read": lambda self: 21.5})
try_build("DigitalThermometer", (Thermometer,), {})
try_build("Barometer", (Sensor,), {"unit": "hPa"})
try_build("Hygrometer", (Sensor,), {})
```

`__init_subclass__` does this job with no metaclass and no exempt-the-base check, since it never runs for `Sensor` itself. Building the test classes with `type()` (Q03) removes four copies of the same `try` block. For required *methods*, `abc.abstractmethod` (NB21) is the standard tool; it complains at instantiation rather than at class creation.

</details>

In [ ]:
# your code here


### NB25-Q29 · L5 · __getattr__ · Predict

A property with a typo, on a class with `__getattr__`. Predict the output.

**What does this print?**

```python
class Profile:
    def __init__(self, first, last):
        self.first = first
        self.last = last

    @property
    def full_name(self):
        return self.first.capitalise() + " " + self.last

    def __getattr__(self, name):
        return f"<no {name}>"


grace = Profile("grace", "Hopper")
print(grace.full_name)
print(grace.first)
```

<details><summary><b>Hint</b></summary>

`capitalise` is a misspelling of `capitalize`, so it raises `AttributeError` inside the property. What does Python do whenever a normal lookup raises `AttributeError`?

</details>

<details><summary><b>Expected output</b></summary>

```text
<no full_name>
grace
```

</details>

<details><summary><b>Solution</b></summary>

`str` has no `capitalise` method, so the property raises `AttributeError`. Python can't tell that apart from "`full_name` doesn't exist", so it falls back to `__getattr__(grace, "full_name")`, which returns `<no full_name>`. The real bug (the typo) is hidden completely, and the error message points at the wrong name.

</details>

<details><summary><b>Senior dev solution</b></summary>

Any `AttributeError` raised inside a property, or inside anything it calls, is swallowed and replaced by a `__getattr__` call. Seniors keep `__getattr__` narrow (raise `AttributeError` for names it doesn't own) so a masked bug at least fails loudly instead of returning a fake value, and a property that does risky lookups can catch `AttributeError` and re-raise it as `RuntimeError` so the real cause surfaces.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB25-Q30 · L5 · Descriptors · Write

Write a decorator class `cached` that works like a lazy property. On the
first read it calls the wrapped function, stores the result in the
instance's `__dict__` under the same name, and returns it. Give it **no**
`__set__`, so later reads find the instance value and never reach the
descriptor again. Use `__set_name__` to learn the name.

Use it on `Report.total`, which prints `computing...` and returns
`sum(self.values)`. Read `total` twice, delete it with `del`, then read
it once more:

```text
computing...
60
60
computing...
60
```

**Starting code (already in the cell below):**

```python
class Report:
    def __init__(self, values):
        self.values = values
```

<details><summary><b>Hint</b></summary>

Only a *non-data* descriptor (no `__set__`) can be hidden by the instance dict (Q11). That's exactly what makes the caching work. You'll need to redefine `Report` with the decorated method.

</details>

<details><summary><b>Expected output</b></summary>

```text
computing...
60
60
computing...
60
```

</details>

<details><summary><b>Solution</b></summary>

```python
class cached:
    def __init__(self, func):
        self.func = func

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        value = self.func(obj)
        obj.__dict__[self.name] = value
        return value


class Report:
    def __init__(self, values):
        self.values = values

    @cached
    def total(self):
        print("computing...")
        return sum(self.values)


report = Report([10, 20, 30])
print(report.total)
print(report.total)
del report.total
print(report.total)
```

The first read finds nothing in the instance dict, so the non-data descriptor runs, computes `60` and saves it as `report.__dict__["total"]`. The second read finds that instance value *first* (instance dict beats non-data descriptors), so nothing is computed. `del report.total` removes the cached value, and the next read reaches the descriptor again. The trap: adding a `__set__` would make it a data descriptor, which always wins, so it would recompute on every read.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import functools


class Report:
    def __init__(self, values):
        self.values = values

    @functools.cached_property
    def total(self):
        print("computing...")
        return sum(self.values)


report = Report([10, 20, 30])
print(report.total)
print(report.total)
del report.total
print(report.total)
```

`functools.cached_property` is this exact descriptor, already tested. The catch for both: the cache never notices when `values` changes, so seniors use it for data that's fixed after creation, and `del` the attribute to invalidate when needed.

</details>

In [ ]:
class Report:
    def __init__(self, values):
        self.values = values

# your code here


### NB25-Q31 · L5 · __class_getitem__ · Write

Write `TypedList`, a subclass of `list` whose `append` only accepts one
type. `TypedList[int]` must return a **subclass** built with `type()`,
named `TypedList[int]`, whose class attribute `item_type` is `int`. The
same subscription must always return the **same** class, so keep the
classes you create in a dict.

`append` raises `TypeError("<class name> cannot hold <type>")` for a wrong
value. Then:

```text
TypedList[int] [3, 4]
True
TypedList[int] cannot hold str
True
```

That's: the class name and the list after appending `3` and `4`; whether
`TypedList[int] is TypedList[int]`; the error from appending `"5"`; and
whether the object is still a `list`.

<details><summary><b>Hint</b></summary>

In `__class_getitem__`, build the class only if it isn't in your dict yet: `type(name, (cls,), {"item_type": item_type})`. In `append`, check the type, then call `super().append(value)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
TypedList[int] [3, 4]
True
TypedList[int] cannot hold str
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class TypedList(list):
    item_type = object
    made = {}

    def __class_getitem__(cls, item_type):
        if item_type not in TypedList.made:
            name = f"TypedList[{item_type.__name__}]"
            TypedList.made[item_type] = type(name, (cls,), {"item_type": item_type})
        return TypedList.made[item_type]

    def append(self, value):
        if not isinstance(value, self.item_type):
            raise TypeError(f"{type(self).__name__} cannot hold {type(value).__name__}")
        super().append(value)


IntList = TypedList[int]
numbers = IntList()
numbers.append(3)
numbers.append(4)
print(IntList.__name__, numbers)
print(TypedList[int] is TypedList[int])
try:
    numbers.append("5")
except TypeError as error:
    print(error)
print(isinstance(numbers, list))
```

`TypedList[int]` calls `__class_getitem__`, which uses three-argument `type()` to build a subclass with `item_type = int`. Without the `made` dict, each `TypedList[int]` would be a brand-new class, so the `is` check would be `False` and `isinstance` against one would fail for objects made from another. `append` reads `item_type` through `self`, so the subclass's value is used, and `super().append` does the real work.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import functools


class TypedList(list):
    item_type = object

    @classmethod
    @functools.cache
    def __class_getitem__(cls, item_type):
        return type(f"{cls.__name__}[{item_type.__name__}]", (cls,),
                    {"item_type": item_type})

    def append(self, value):
        if not isinstance(value, self.item_type):
            raise TypeError(f"{type(self).__name__} cannot hold {type(value).__name__}")
        super().append(value)


IntList = TypedList[int]
numbers = IntList()
numbers.append(3)
numbers.append(4)
print(IntList.__name__, numbers)
print(TypedList[int] is TypedList[int])
try:
    numbers.append("5")
except TypeError as error:
    print(error)
print(isinstance(numbers, list))
```

`functools.cache` is the memo dict, keyed on both `cls` and `item_type`. The explicit `@classmethod` is needed because Python only auto-converts a *plain function*. The bigger catch: `extend`, `insert`, `+=` and `list(...)` all skip `append`, so a real typed list subclasses `collections.UserList` (NB21) or checks in every mutating method.

</details>

In [ ]:
# your code here


### NB25-Q32 · L5 · Metaclasses · Error

Two libraries, two metaclasses, one class that uses both. Which error does this raise?

**Which error does this raise?**

```python
class PluginMeta(type):
    pass


class ModelMeta(type):
    pass


class Plugin(metaclass=PluginMeta):
    pass


class Model(metaclass=ModelMeta):
    pass


class AuditedModel(Model, Plugin):
    pass


print("created")
```

<details><summary><b>Hint</b></summary>

A class has exactly one metaclass. Which one should `AuditedModel` get?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError` (a *metaclass conflict*). A class's metaclass must be a subclass of the metaclasses of all its bases. `ModelMeta` and `PluginMeta` are unrelated, so Python can't choose one that satisfies both, and refuses to build the class.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class PluginMeta(type):
    pass


class ModelMeta(type):
    pass


class Plugin(metaclass=PluginMeta):
    pass


class Model(metaclass=ModelMeta):
    pass


class AuditedModelMeta(ModelMeta, PluginMeta):
    pass


class AuditedModel(Model, Plugin, metaclass=AuditedModelMeta):
    pass


print("created")
```

The fix is a combined metaclass that inherits from both, and it only works if the two cooperate via `super()`. This conflict is the strongest practical argument against metaclasses in libraries: mixing an `ABC` (whose metaclass is `ABCMeta`) with a framework model hits it all the time. `__init_subclass__` never conflicts.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## Mini-project

Pull together everything from this notebook.

## NB25-P · L5 · Mini-project: Mini ORM · Write

Build the core of a tiny ORM (object-relational mapper).

- `Field(kind, required=True)`: a data descriptor. `__set_name__` stores
  the owner class and the field name. `__set__` raises
  `TypeError("<Model>.<field> must be <kind>, got <type>")` for a wrong
  type and stores the value in the instance `__dict__` otherwise.
  `__get__` returns the descriptor when read from the class, and the
  stored value (or `None` if never set) from an instance.
- `Model`: the base class. Its `__init_subclass__(cls, table, **kwargs)`
  stores `Model.registry[table] = cls` and sets `cls.fields` to the list
  of the class's `Field` names, in definition order.
- `Model.__init__(self, **values)`: an unknown keyword raises
  `TypeError("<Model> has no field <name>")`. Missing required fields
  raise `TypeError("<Model> missing: <names>")`. Otherwise set each value
  (so types are checked).
- `Model.__repr__`: `<Model>(field=value, ...)` with `repr` values, every
  field included.

Models: `Customer` (table `customers`) with `id` (int), `name` (str) and
an optional `email` (str); `Order` (table `orders`) with `id` (int),
`customer_id` (int) and `total` (float).

Then print the sorted table names and the `Order` fields, create
`Customer(id=1, name="Ada")` and `Order(id=10, customer_id=1,
total=25.5)` and print both, and try three bad records, printing
`Error: <message>` for each: an order with `total="9.99"`, a customer
with only `id=2`, and a customer with an extra `phone="555"`. Finally
print whether `Model.registry["orders"]` is `Order`:

```text
Tables: ['customers', 'orders']
Order fields: id, customer_id, total
Customer(id=1, name='Ada', email=None)
Order(id=10, customer_id=1, total=25.5)
Error: Order.total must be float, got str
Error: Customer missing: name
Error: Customer has no field phone
True
```

<details><summary><b>Hint</b></summary>

`vars(cls)` keeps the class body's definition order, so loop over its items and keep names whose value is a `Field`. Check unknown names first, then missing required ones, then set the values with `setattr`. For the bad records, put three `lambda`s in a list and call each inside `try`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Tables: ['customers', 'orders']
Order fields: id, customer_id, total
Customer(id=1, name='Ada', email=None)
Order(id=10, customer_id=1, total=25.5)
Error: Order.total must be float, got str
Error: Customer missing: name
Error: Customer has no field phone
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Field:
    def __init__(self, kind, required=True):
        self.kind = kind
        self.required = required

    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        if not isinstance(value, self.kind):
            raise TypeError(f"{self.owner.__name__}.{self.name} must be "
                            f"{self.kind.__name__}, got {type(value).__name__}")
        obj.__dict__[self.name] = value


class Model:
    registry = {}

    def __init_subclass__(cls, table, **kwargs):
        super().__init_subclass__(**kwargs)
        Model.registry[table] = cls
        cls.fields = []
        for name, value in vars(cls).items():
            if isinstance(value, Field):
                cls.fields.append(name)

    def __init__(self, **values):
        model = type(self).__name__
        for name in values:
            if name not in self.fields:
                raise TypeError(f"{model} has no field {name}")
        missing = []
        for name in self.fields:
            if vars(type(self))[name].required and name not in values:
                missing.append(name)
        if missing:
            raise TypeError(f"{model} missing: {', '.join(missing)}")
        for name, value in values.items():
            setattr(self, name, value)

    def __repr__(self):
        parts = []
        for name in self.fields:
            parts.append(f"{name}={getattr(self, name)!r}")
        return f"{type(self).__name__}({', '.join(parts)})"


class Customer(Model, table="customers"):
    id = Field(int)
    name = Field(str)
    email = Field(str, required=False)


class Order(Model, table="orders"):
    id = Field(int)
    customer_id = Field(int)
    total = Field(float)


print("Tables:", sorted(Model.registry))
print("Order fields:", ", ".join(Order.fields))
ada = Customer(id=1, name="Ada")
print(ada)
print(Order(id=10, customer_id=ada.id, total=25.5))

attempts = [
    lambda: Order(id=11, customer_id=1, total="9.99"),
    lambda: Customer(id=2),
    lambda: Customer(id=3, name="Bo", phone="555"),
]
for attempt in attempts:
    try:
        attempt()
    except TypeError as error:
        print("Error:", error)
print(Model.registry["orders"] is Order)
```

Each `Field` is a data descriptor, so every assignment, including the `setattr` calls in `__init__`, is type-checked. `__set_name__` runs before `__init_subclass__` while the class is built, so by the time the base registers a model every field already knows its owner and name. `vars(cls)` is the class's own namespace in definition order, which gives `fields`. `__init__` checks unknown and missing names *before* setting anything, so a bad record never ends up half-built, and `__repr__` is written once for every model.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Field:
    def __init__(self, kind, *, required=True):
        self.kind = kind
        self.required = required

    def __set_name__(self, owner, name):
        self.owner, self.name = owner, name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        if not isinstance(value, self.kind):
            raise TypeError(f"{self.owner.__name__}.{self.name} must be "
                            f"{self.kind.__name__}, got {type(value).__name__}")
        obj.__dict__[self.name] = value


class Model:
    registry = {}

    def __init_subclass__(cls, *, table, **kwargs):
        super().__init_subclass__(**kwargs)
        if table in Model.registry:
            raise TypeError(f"table {table!r} is already mapped")
        Model.registry[table] = cls
        cls.table = table
        cls.fields = {name: value for name, value in vars(cls).items()
                      if isinstance(value, Field)}

    def __init__(self, **values):
        model = type(self).__name__
        if unknown := values.keys() - self.fields.keys():
            raise TypeError(f"{model} has no field {', '.join(sorted(unknown))}")
        missing = [name for name, field in self.fields.items()
                   if field.required and name not in values]
        if missing:
            raise TypeError(f"{model} missing: {', '.join(missing)}")
        for name, value in values.items():
            setattr(self, name, value)

    def __repr__(self):
        body = ", ".join(f"{name}={getattr(self, name)!r}" for name in self.fields)
        return f"{type(self).__name__}({body})"


class Customer(Model, table="customers"):
    id = Field(int)
    name = Field(str)
    email = Field(str, required=False)


class Order(Model, table="orders"):
    id = Field(int)
    customer_id = Field(int)
    total = Field(float)


print("Tables:", sorted(Model.registry))
print("Order fields:", ", ".join(Order.fields))
ada = Customer(id=1, name="Ada")
print(ada)
print(Order(id=10, customer_id=ada.id, total=25.5))

bad_records = [
    lambda: Order(id=11, customer_id=1, total="9.99"),
    lambda: Customer(id=2),
    lambda: Customer(id=3, name="Bo", phone="555"),
]
for make in bad_records:
    try:
        make()
    except TypeError as error:
        print("Error:", error)
print(Model.registry["orders"] is Order)
```

Keeping `fields` as a dict of name to `Field` saves looking the descriptor up again, and dict key views support set maths (`values.keys() - fields.keys()`) to find unknown names in one step. Keyword-only `table` and `required` make call sites self-explanatory, and the duplicate-table check catches copy-paste errors at import. Django and SQLAlchemy still use metaclasses for this, mostly for history; `__init_subclass__` covers it today, and `typing.dataclass_transform` (NB26) tells type checkers what the generated `__init__` accepts.

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB26 · Type Hints** in `advanced/`.